# Notebook 01 — Harvest activations from Pythia-160M

Phase 1 of the NLA reimplementation. We:

1. Load `EleutherAI/pythia-160m`.
2. Register a forward hook on **layer 6** (middle of 12) of the residual stream.
3. Stream snippets from `wikitext-2-raw-v1`, truncate each to a random length, and capture the **last-token residual activation**.
4. Save the activations (`.pt`) and matching source-text metadata (`.jsonl`) into `data/activations/`.

These activations become the **dataset** that the reconstructor will learn to invert and that the verbalizer will learn to describe.

Open this notebook in Colab: **Runtime → Change runtime type → T4 GPU**, then run all cells.

## 0. Environment

In [ ]:
# Skip the install if running locally with deps already present.
# Note: sympy is upgraded because Colab's GPU runtime ships with an
# older version that crashes when PyTorch imports torch._dynamo.
%pip install -q transformers==4.46.0 datasets==3.1.0 accelerate matplotlib "sympy>=1.13.3"

In [ ]:
import json, os, random
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# Persistent storage: mount Google Drive on Colab so saved artifacts survive
# session resets. On local Python, just use the project root.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = Path('/content/drive/MyDrive/kth-nla-2026')
    print(f'persistent storage: {DATA_ROOT}')
except ImportError:
    DATA_ROOT = Path('.')
    print(f'local storage: {DATA_ROOT.resolve()}')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

## 1. Load the target model

Pythia-160M is a 12-layer GPT-NeoX-style decoder with `hidden_size=768`. We're going to grab activations at **layer 6**— middle of the stack, matching the paper's 'middle-to-late layer' choice in spirit while keeping reconstructor depth manageable.

In [ ]:
MODEL_NAME = 'EleutherAI/pythia-160m'
LAYER = 6  # 0-indexed; layer 6 of 12

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device).eval()
for p in model.parameters():
    p.requires_grad_(False)

cfg = model.config
print(f'model: {MODEL_NAME}')
print(f'n_layers: {cfg.num_hidden_layers}, hidden_size: {cfg.hidden_size}, n_heads: {cfg.num_attention_heads}')
print(f'vocab_size: {cfg.vocab_size}')

## 2. Hook layer 6

We attach a forward hook to the layer-6 transformer block. Its output is a tuple; index 0 is the residual stream of shape `(batch, seq, hidden)`. We stash it in a global dict and read out only the **last-token** vector after each forward pass.

In [ ]:
captured = {}

def hook_fn(module, inputs, output):
    h = output[0] if isinstance(output, tuple) else output  # (B, T, D)
    captured['h'] = h.detach()

layer_module = model.gpt_neox.layers[LAYER]
handle = layer_module.register_forward_hook(hook_fn)

# Sanity check: one short string -> one activation of shape (hidden,).
test_text = 'The cat sat on the mat and looked out the window.'
ids = tok(test_text, return_tensors='pt').input_ids.to(device)
with torch.no_grad():
    _ = model(ids)
h = captured['h'][0, -1, :]
print('activation shape:', tuple(h.shape))
print(f'mean={h.mean().item():+.4f}  std={h.std().item():.4f}  L2_norm={h.norm().item():.4f}')

## 3. Text source: WikiText-2

Small, classic, easy to load. Each retained line is at least 200 characters so we get substantive snippets, not empty lines or article titles.

In [ ]:
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
texts = [t for t in ds['text'] if len(t.strip()) > 200]
print(f'snippets after filtering: {len(texts):,}')
print('example:', texts[0][:160].replace('\n', ' ') + ' ...')

## 4. Harvest activations

For each sample: pick a random snippet, tokenize, randomly truncate to a length in `[MIN_TOK, MAX_TOK]`, run a forward pass, and capture the last-token activation at layer 6. We also record the truncated text — it'll be useful later as ground-truth context for warm-starting the verbalizer.

On a Colab T4 this should take roughly 1–2 minutes for `N=5000`.

In [ ]:
N = 5000
MIN_TOK, MAX_TOK = 16, 256

acts = []
records = []
i = 0
while len(acts) < N:
    text = random.choice(texts)
    full_ids = tok(text, return_tensors='pt', truncation=True, max_length=512).input_ids[0]
    if full_ids.numel() < MIN_TOK + 2:
        continue
    cut = random.randint(MIN_TOK, min(MAX_TOK, full_ids.numel()))
    ids = full_ids[:cut].unsqueeze(0).to(device)
    with torch.no_grad():
        _ = model(ids)
    h = captured['h'][0, -1, :].cpu().to(torch.float32)
    acts.append(h)
    records.append({
        'idx': len(acts) - 1,
        'n_tokens': int(cut),
        'snippet': tok.decode(full_ids[:cut]),
    })
    i += 1
    if len(acts) % 500 == 0:
        print(f'collected {len(acts):,} / {N:,}')

acts_tensor = torch.stack(acts)  # (N, hidden_size)
print('done. activations tensor:', tuple(acts_tensor.shape), acts_tensor.dtype)

## 5. Save

Activations as `.pt` (efficient binary), metadata as `.jsonl` (one record per line, human-readable).

In [ ]:
out_dir = DATA_ROOT / 'data/activations'
out_dir.mkdir(parents=True, exist_ok=True)
stem = f'pythia160m_layer{LAYER}_wikitext2_{len(acts_tensor)}'

torch.save(acts_tensor, out_dir / f'{stem}.pt')
with open(out_dir / f'{stem}.jsonl', 'w', encoding='utf-8') as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print('saved:')
print(' ', out_dir / f'{stem}.pt')
print(' ', out_dir / f'{stem}.jsonl')

## 6. Sanity checks

Confirm the shape and distribution look reasonable. Of particular interest: **the raw variance** — this is the denominator of our FVE metric, so we want it to be meaningfully large (not collapsed to ~0).

In [ ]:
import matplotlib.pyplot as plt

mean_act = acts_tensor.mean(0)
var_total = ((acts_tensor - mean_act) ** 2).sum(dim=1).mean().item()
print(f'N = {acts_tensor.shape[0]}, d_model = {acts_tensor.shape[1]}')
print(f'global mean magnitude:  {mean_act.norm().item():.4f}')
print(f'mean L2 norm per act:   {acts_tensor.norm(dim=1).mean().item():.4f}')
print(f'RAW total variance (FVE denominator): {var_total:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(acts_tensor.norm(dim=1).numpy(), bins=50)
axes[0].set_title('L2 norm of harvested activations')
axes[0].set_xlabel('||h||'); axes[0].set_ylabel('count')
axes[1].hist(acts_tensor.flatten().numpy(), bins=80)
axes[1].set_title('Per-coordinate value distribution')
axes[1].set_xlabel('h_i'); axes[1].set_ylabel('count')
plt.tight_layout(); plt.show()

## 7. (If running on Colab) commit results back to GitHub

Optional: push the `data/activations/` files to the private repo so the next notebook can pick them up. Skip if you'd rather download manually.

In [ ]:
# Uncomment + edit if needed.
# !git config --global user.email 'sppooja225@gmail.com'
# !git config --global user.name 'Pooja Sollapura'
# !git add data/activations/
# !git commit -m 'Harvest 5k Pythia-160M layer-6 activations on WikiText-2'
# !git push